# MP2 Part 1 — commit retrieval for aamphone's 10 projects
Retrieves every commit for each assigned project from World of Code (WoC V2412) and writes `aamphone_project_summary.csv` (semicolon-separated: project_wocid; commit_sha1; author; time; commit message).

In [ ]:
# python-woc has no prebuilt package for Python 3.13 (Colab's current version), so
# `pip install python-woc` tries to compile it and fails. Its remote API client is plain
# Python, so load just that part straight from the official source archive instead.
!pip install -q httpx tqdm pandas matplotlib
!wget -q -O python_woc.tar.gz https://files.pythonhosted.org/packages/d9/20/b345ed79a5ee3d484d8133aef8e7d5219aefcde0e578a60f3ee6467eb65d/python_woc-0.4.1.tar.gz
!tar xzf python_woc.tar.gz
!mkdir -p woc_pkg/woc && cp python_woc-0.4.1/woc/remote.py python_woc-0.4.1/woc/base.py woc_pkg/woc/ && echo '__version__ = "0.4.1"' > woc_pkg/woc/__init__.py
# please clear the output of this cell in your notebook before checking it in

In [ ]:
import sys
sys.path.insert(0, 'woc_pkg')  # the WoC client unpacked by the cell above
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
import time

netid = 'aamphone'
woc = WocMapsRemote(base_url="https://worldofcode.org/api/")
# woc = WocMapsRemote(base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY")

# Assigned projects from net2prj.csv (netID == aamphone): WoC id -> GitHub URL
projects = {
    'arviz-devs_arviz': 'https://github.com/arviz-devs/arviz',
    'clindatsci_dicomweb-python-client': 'https://github.com/clindatsci/dicomweb-python-client',
    'peal_vole': 'https://github.com/peal/vole',
    'nbara_python-meegkit': 'https://github.com/nbara/python-meegkit',
    'flyaflya_causact': 'https://github.com/flyaflya/causact',
    'geolytix_xyz': 'https://github.com/GEOLYTIX/xyz',
    'genouest_biomaj-download': 'https://github.com/genouest/biomaj-download',
    'pressio_pressio': 'https://github.com/Pressio/pressio',
    'kkondo1981_aglm': 'https://github.com/kkondo1981/aglm',
    'winvector_vtreat': 'https://github.com/WinVector/vtreat',
}

## Step 1 — commit SHA1s per project (p2c map)

In [ ]:
list_df_commits = []
for prj in projects:
    for attempt in range(3):
        try:
            commits = woc.get_values('p2c', prj)
            break
        except Exception as e:
            print(prj, 'attempt', attempt + 1, 'failed:', e)
            time.sleep(5)
    else:
        raise RuntimeError('Could not retrieve commits for ' + prj)
    d = pd.DataFrame(commits, columns=['sha1'])
    d['project'] = prj
    print(f'{prj}: {len(d)} commits')
    list_df_commits.append(d)
    time.sleep(1)

df = pd.concat(list_df_commits, ignore_index=True)
df.to_csv('df_commits.csv', index=False)  # saved so the SHA1 list need not be fetched again
df.groupby('project').size()

## Step 2 — commit content in batches
Batches of 10 SHA1s (below the 50 limit) with a 1-second wait between requests. Failed batches are retried, and any SHA1 still missing is reported at the end.

In [ ]:
# a commit can belong to several projects, so retrieve each unique SHA1 once
shas = df['sha1'].drop_duplicates().tolist()
chunks = [shas[x:x + 10] for x in range(0, len(shas), 10)]
commit_content = {}
failed = []

for chunk in tqdm(chunks):
    for attempt in range(3):
        try:
            res, err = woc.get_values_many('commit.tch', chunk)
            break
        except Exception as e:
            print('retrying after error:', e)
            time.sleep(5 * (attempt + 1))
    else:
        failed.extend(chunk)
        continue
    if err:
        print('Got Errors', err)
    for k, v in res.items():
        commit_content[k] = v[0]  # needed because of the internal implementation
    time.sleep(1)  # stay under the rate limit

missing = [s for s in shas if s not in commit_content]
print('retrieved', len(commit_content), 'of', len(shas), 'unique commits; missing:', len(missing))

In [ ]:
# flatten: commit = [tree, [parents], [author, unixtime, tz], [committer, unixtime, tz], message]
commit_data = []
for sha, commit in commit_content.items():
    commit_data.append({
        'commit': sha,
        'tree': commit[0],
        'parent': list(commit[1]),
        'author': commit[2][0],
        'author_time': int(commit[2][1]),
        'author_tz': commit[2][2],
        'committer': commit[3][0],
        'committer_time': int(commit[3][1]),
        'committer_tz': commit[3][2],
        'message': commit[4],
    })
df_commit_data = pd.DataFrame(commit_data).merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
print(commit_data[0])
df_commit_data.head(2)

## Step 3 — write aamphone_project_summary.csv
One row per (project, commit). The project column comes from each commit's own project (the template reused the loop variable `prj`, which would label every row with the last project). The file is written once in `w` mode so re-running the cell does not duplicate rows.

In [ ]:
dfinf = df_commit_data[['project', 'commit', 'author', 'author_time', 'message']].rename(columns={'author_time': 'time'})
dfinf = dfinf.sort_values(['project', 'time']).reset_index(drop=True)
dfinf.to_csv(netid + '_project_summary.csv', index=False, sep=';', mode='w', header=False)
dfinf.head(1)

## Step 4 — WoC statistics per project

In [ ]:
woc_stats = dfinf.groupby('project').agg(
    ncommits=('commit', 'nunique'),
    nauthors=('author', 'nunique'),
    min_time=('time', 'min'),
    max_time=('time', 'max'),
)
woc_stats['from'] = pd.to_datetime(woc_stats['min_time'], unit='s').dt.strftime('%Y-%m-%d')
woc_stats['to'] = pd.to_datetime(woc_stats['max_time'], unit='s').dt.strftime('%Y-%m-%d')
woc_stats

## GitHub data (collected 2026-10-07)
Stars and forks as displayed on each repository page (GitHub rounds counts above 1,000, so arviz shows 1.8k). Last commit date is the newest commit on the default branch.

| Project | GitHub repo | Stars | Forks | Last commit date |
|---|---|---|---|---|
| arviz-devs_arviz | arviz-devs/arviz | 1.8k | 501 | 2026-09-09 |
| clindatsci_dicomweb-python-client | clindatsci/dicomweb-python-client (now redirects to ImagingDataCommons/dicomweb-client) | 129 | 42 | 2026-08-17 |
| peal_vole | peal/vole | 8 | 2 | 2026-07-01 |
| nbara_python-meegkit | nbara/python-meegkit | 227 | 57 | 2026-08-21 |
| flyaflya_causact | flyaflya/causact | 47 | 12 | 2025-09-12 |
| geolytix_xyz | GEOLYTIX/xyz | 132 | 44 | 2026-10-02 |
| genouest_biomaj-download | genouest/biomaj-download | 1 | 7 | 2025-05-20 |
| pressio_pressio | Pressio/pressio (default branch: develop) | 50 | 7 | 2025-09-10 |
| kkondo1981_aglm | kkondo1981/aglm | 17 | 5 | 2025-05-12 |
| winvector_vtreat | WinVector/vtreat | 284 | 45 | 2025-01-09 |


## WoC data (World of Code API, retrieved 2026-10-08)
Number of commits, number of distinct author strings, and min/max author time (Unix seconds, with the UTC date).

| Project | Commits | Authors | Min time | Max time |
|---|---|---|---|---|
| arviz-devs_arviz | 9,492 | 395 | 1438170670 (2015-07-29) | 1761135327 (2025-10-22) |
| clindatsci_dicomweb-python-client | 518 | 39 | 1521487507 (2018-03-19) | 1757491841 (2025-09-10) |
| peal_vole | 595 | 7 | 1395418492 (2014-03-21) | 1757678214 (2025-09-12) |
| nbara_python-meegkit | 479 | 45 | 1227075592 (2008-11-19) | 1775048948 (2026-04-01) |
| flyaflya_causact | 457 | 21 | 1524067862 (2018-04-18) | 1757705466 (2025-09-12) |
| geolytix_xyz | 15,306 | 65 | 1512569316 (2017-12-06) | 1762277501 (2025-11-04) |
| genouest_biomaj-download | 505 | 19 | 1472208351 (2016-08-26) | 1750133502 (2025-06-17) |
| pressio_pressio | 4,282 | 52 | 1525121445 (2018-04-30) | 1757514162 (2025-09-10) |
| kkondo1981_aglm | 366 | 6 | 1546232994 (2018-12-31) | 1746979099 (2025-05-11) |
| winvector_vtreat | 1,081 | 6 | 1407432287 (2014-08-07) | 1736461287 (2025-01-09) |

Total: 33,081 rows. 33,140 unique commit SHA1s were listed by `p2c`; 59 of them returned "Key … not found" from WoC's commit store, so 33,081 are in `aamphone_project_summary.csv`.

**Validation against GitHub.** WoC counts are much higher than GitHub's default-branch counts (e.g., arviz 9,492 vs 1,670 on `main`; geolytix_xyz 15,306 vs 6,217) because WoC's `p2c` includes commits from all branches and from forks it attributes to the project. Author counts are also inflated because the same person appears under several name/e-mail strings (e.g., `aloctavodia` and `Osvaldo Martin`).


# Make sure you check in to your fork not just the notebook but also the csv files!!!